# Food-101: classification with EfficientNet-B0

**Dataset path:** set environment variable `FOOD101_IMAGES_DIR` to your Food-101 `images` folder (the directory that contains one subfolder per class), or place data at `data/food-101/images` or `food-101/images`. On **Kaggle**, the notebook still searches under `/kaggle/input` when those are not found.

The notebook includes exploratory cells and repeated setup blocks from iterative runs—**run from the top** for a coherent flow.

In [ ]:
import os
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, UnidentifiedImageError

In [ ]:
# -----------------------------------
# 1. Locate the correct images folder
# -----------------------------------

def find_food101_images_dir(base_dir="/kaggle/input"):
    """Resolve Food-101 `images` folder: env FOOD101_IMAGES_DIR, local guesses, then Kaggle search."""
    override = os.environ.get("FOOD101_IMAGES_DIR")
    if override:
        p = Path(override).expanduser().resolve()
        if p.is_dir():
            n_sub = sum(1 for x in p.iterdir() if x.is_dir())
            if n_sub >= 50:
                return p
        raise FileNotFoundError(
            f"FOOD101_IMAGES_DIR is set but is not a valid Food-101 images directory: {p}"
        )

    cwd = Path.cwd()
    for rel in (
        cwd / "data" / "food-101" / "images",
        cwd / "food-101" / "images",
        Path("data/food-101/images"),
        Path("food-101/images"),
    ):
        try:
            r = rel.resolve()
        except OSError:
            continue
        if r.is_dir():
            subdirs = [x for x in r.iterdir() if x.is_dir()]
            if len(subdirs) >= 50:
                return r

    base = Path(base_dir)
    if not base.is_dir():
        raise FileNotFoundError(
            "Could not find Food-101 images. Set FOOD101_IMAGES_DIR to the directory that "
            "contains class subfolders, place data at data/food-101/images, or run on Kaggle "
            "with the dataset attached."
        )

    candidates = list(base.rglob("images"))
    good_candidates = []
    for c in candidates:
        c_str = str(c).lower()
        if "food" in c_str and c.is_dir():
            subdirs = [x for x in c.iterdir() if x.is_dir()]
            if len(subdirs) >= 50:
                good_candidates.append(c)
    if not good_candidates:
        raise FileNotFoundError(
            "Could not automatically find Food-101 images directory under /kaggle/input."
        )
    return sorted(good_candidates, key=lambda x: len(str(x)))[0]

images_dir = find_food101_images_dir()
print("Images directory found:")
print(images_dir)

In [ ]:
# -----------------------------------
# 2. Inspect folder structure
# -----------------------------------

class_names = sorted([d.name for d in images_dir.iterdir() if d.is_dir()])

print(f"Number of classes: {len(class_names)}")
print("First 10 classes:", class_names[:10])
print("Last 10 classes:", class_names[-10:])

In [ ]:
# -----------------------------------
# 2. Get class names
# -----------------------------------

class_names = sorted([d.name for d in images_dir.iterdir() if d.is_dir()])

print(f"Number of classes: {len(class_names)}")
print("First 20 classes:")
print(class_names[:20])

In [ ]:
# -----------------------------------
# 3. Count images per class
# -----------------------------------

import pandas as pd

rows = []

for cls in class_names:
    cls_dir = images_dir / cls
    image_files = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]]
    
    rows.append({
        "class_name": cls,
        "image_count": len(image_files)
    })

class_df = pd.DataFrame(rows)

print(class_df.head())
print("\nTotal images:", class_df["image_count"].sum())

In [ ]:
# -----------------------------------
# 4. Plot class distribution
# -----------------------------------

import matplotlib.pyplot as plt

plt.figure(figsize=(20, 6))
plt.bar(class_df["class_name"], class_df["image_count"])
plt.title("Number of Images per Food Class")
plt.xlabel("Food Class")
plt.ylabel("Image Count")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------
# 5. Visualise random sample images
# -----------------------------------

import random
from PIL import Image

sample_classes = random.sample(class_names, 9)

plt.figure(figsize=(14, 14))

for i, cls in enumerate(sample_classes, 1):
    cls_dir = images_dir / cls
    image_files = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]]
    
    img_path = random.choice(image_files)
    img = Image.open(img_path)
    
    plt.subplot(3, 3, i)
    plt.imshow(img)
    plt.title(cls)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------
# 6. Visualise one class in more detail
# -----------------------------------

chosen_class = random.choice(class_names)
cls_dir = images_dir / chosen_class
image_files = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]]

sample_images = random.sample(image_files, 6)

plt.figure(figsize=(12, 8))

for i, img_path in enumerate(sample_images, 1):
    img = Image.open(img_path)
    
    plt.subplot(2, 3, i)
    plt.imshow(img)
    plt.title(chosen_class)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------
# 7. Check image width and height
# -----------------------------------

from PIL import Image
import numpy as np

widths = []
heights = []

for cls in random.sample(class_names, min(20, len(class_names))):
    cls_dir = images_dir / cls
    image_files = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]]
    
    for img_path in image_files[:30]:
        try:
            img = Image.open(img_path)
            widths.append(img.width)
            heights.append(img.height)
        except:
            pass

print("Number of images checked:", len(widths))
print("Min width:", min(widths), "Max width:", max(widths))
print("Min height:", min(heights), "Max height:", max(heights))

In [ ]:
# -----------------------------------
# 8. Plot image width distribution
# -----------------------------------

plt.figure(figsize=(10, 5))
plt.hist(widths, bins=30)
plt.title("Image Width Distribution")
plt.xlabel("Width")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# -----------------------------------
# 9. Plot image height distribution
# -----------------------------------

plt.figure(figsize=(10, 5))
plt.hist(heights, bins=30)
plt.title("Image Height Distribution")
plt.xlabel("Height")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# -----------------------------------
# 10. Top 20 classes by image count
# -----------------------------------

top20 = class_df.sort_values("image_count", ascending=False).head(20)

plt.figure(figsize=(12, 6))
plt.bar(top20["class_name"], top20["image_count"])
plt.title("Top 20 Classes by Image Count")
plt.xlabel("Food Class")
plt.ylabel("Image Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------
# 11. Quick summary
# -----------------------------------

print("Dataset summary")
print("-" * 40)
print("Images directory:", images_dir)
print("Number of classes:", len(class_names))
print("Total images:", class_df["image_count"].sum())
print("Min images in a class:", class_df["image_count"].min())
print("Max images in a class:", class_df["image_count"].max())
print("Average images per class:", round(class_df["image_count"].mean(), 2))

In [ ]:
import pandas as pd
from pathlib import Path

rows = []

class_names = sorted([d.name for d in images_dir.iterdir() if d.is_dir()])

for cls in class_names:
    cls_dir = images_dir / cls
    image_files = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]]
    
    for img_path in image_files:
        rows.append({
            "filepath": str(img_path),
            "label": cls
        })

df = pd.DataFrame(rows)

print(df.shape)
df.head()

In [ ]:
label_to_index = {label: idx for idx, label in enumerate(class_names)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

df["label_idx"] = df["label"].map(label_to_index)

print(df.head())
print("Number of classes:", len(label_to_index))

In [ ]:
print(df["label"].nunique())
print(df["label_idx"].nunique())

In [ ]:
from tqdm import tqdm

rows = []

for cls in tqdm(class_names, desc="Processing classes"):
    cls_dir = images_dir / cls
    image_files = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]]
    
    for img_path in image_files:
        rows.append({
            "filepath": str(img_path),
            "label": cls
        })

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label_idx"],
    random_state=42
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTrain class count sample:")
print(train_df["label"].value_counts().head())

print("\nValidation class count sample:")
print(val_df["label"].value_counts().head())

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    image = tf.io.read_file(filepath)                 # read file from disk
    image = tf.image.decode_jpeg(image, channels=3)  # turn jpeg into image tensor
    image = tf.image.resize(image, IMG_SIZE)         # resize to 224x224
    image = tf.cast(image, tf.float32) / 255.0       # scale pixels to 0-1
    return image, label

# Create raw TensorFlow datasets from your dataframes
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["filepath"].values, train_df["label_idx"].values)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["filepath"].values, val_df["label_idx"].values)
)

# Apply image loading + batching pipeline
train_ds = (
    train_ds
    .shuffle(1000)
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(train_ds)
print(val_ds)

In [ ]:
import matplotlib.pyplot as plt

for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)

    plt.figure(figsize=(12, 12))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(index_to_label[int(labels[i].numpy())])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

# Load pretrained model
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

# Freeze it (VERY important initially)
base_model.trainable = False

# Build final model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
class_names = sorted([d.name for d in images_dir.iterdir() if d.is_dir()])

In [ ]:
import os
from pathlib import Path

def find_food101_images_dir(base_dir="/kaggle/input"):
    """Resolve Food-101 `images` folder: env FOOD101_IMAGES_DIR, local guesses, then Kaggle search."""
    override = os.environ.get("FOOD101_IMAGES_DIR")
    if override:
        p = Path(override).expanduser().resolve()
        if p.is_dir():
            n_sub = sum(1 for x in p.iterdir() if x.is_dir())
            if n_sub >= 50:
                return p
        raise FileNotFoundError(
            f"FOOD101_IMAGES_DIR is set but is not a valid Food-101 images directory: {p}"
        )

    cwd = Path.cwd()
    for rel in (
        cwd / "data" / "food-101" / "images",
        cwd / "food-101" / "images",
        Path("data/food-101/images"),
        Path("food-101/images"),
    ):
        try:
            r = rel.resolve()
        except OSError:
            continue
        if r.is_dir():
            subdirs = [x for x in r.iterdir() if x.is_dir()]
            if len(subdirs) >= 50:
                return r

    base = Path(base_dir)
    if not base.is_dir():
        raise FileNotFoundError(
            "Could not find Food-101 images. Set FOOD101_IMAGES_DIR to the directory that "
            "contains class subfolders, place data at data/food-101/images, or run on Kaggle "
            "with the dataset attached."
        )

    candidates = list(base.rglob("images"))
    good_candidates = []
    for c in candidates:
        c_str = str(c).lower()
        if "food" in c_str and c.is_dir():
            subdirs = [x for x in c.iterdir() if x.is_dir()]
            if len(subdirs) >= 50:
                good_candidates.append(c)
    if not good_candidates:
        raise FileNotFoundError(
            "Could not automatically find Food-101 images directory under /kaggle/input."
        )
    return sorted(good_candidates, key=lambda x: len(str(x)))[0]

images_dir = find_food101_images_dir()
print(images_dir)

In [ ]:
from pathlib import Path

images_dir = Path("/kaggle/input/datasets/dansbecker/food-101/food-101/food-101/images")

print(images_dir)
print(images_dir.exists())

In [ ]:
class_names = sorted([d.name for d in images_dir.iterdir() if d.is_dir()])

print("Number of classes:", len(class_names))
print(class_names[:10])

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)


In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["filepath"].values, train_df["label_idx"].values)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["filepath"].values, val_df["label_idx"].values)
)

train_ds = (
    train_ds
    .shuffle(1000)
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(train_ds)
print(val_ds)

In [ ]:
from pathlib import Path

images_dir = Path("/kaggle/input/datasets/dansbecker/food-101/food-101/food-101/images")

In [ ]:
class_names = sorted([d.name for d in images_dir.iterdir() if d.is_dir()])

In [ ]:
import pandas as pd

rows = []

for cls in class_names:
    cls_dir = images_dir / cls
    for img_path in cls_dir.iterdir():
        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            rows.append({
                "filepath": str(img_path),
                "label": cls
            })

df = pd.DataFrame(rows)

In [ ]:
label_to_index = {label: idx for idx, label in enumerate(class_names)}
df["label_idx"] = df["label"].map(label_to_index)

In [ ]:
import os
from pathlib import Path

import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

In [43]:
import os
from pathlib import Path

import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

In [44]:
images_dir = Path("/kaggle/input/datasets/dansbecker/food-101/food-101/food-101/images")

print("Images path exists:", images_dir.exists())

class_names = sorted([d.name for d in images_dir.iterdir() if d.is_dir()])

print("Number of classes:", len(class_names))
print("First 10 classes:", class_names[:10])

Images path exists: True
Number of classes: 101
First 10 classes: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito']


In [45]:
rows = []

for cls in class_names:
    cls_dir = images_dir / cls
    for img_path in cls_dir.iterdir():
        if img_path.is_file() and img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
            rows.append({
                "filepath": str(img_path),
                "label": cls
            })

df = pd.DataFrame(rows)

print("Dataframe shape:", df.shape)
print(df.head())

KeyboardInterrupt: 

In [46]:
from tqdm.auto import tqdm
import pandas as pd

valid_exts = {".jpg", ".jpeg", ".png", ".webp"}
rows = []

for cls in tqdm(class_names, desc="Processing classes"):
    cls_dir = images_dir / cls
    image_files = [p for p in cls_dir.iterdir() if p.suffix.lower() in valid_exts]
    rows.extend([{"filepath": str(p), "label": cls} for p in image_files])

df = pd.DataFrame(rows)

print("Dataframe shape:", df.shape)
print(df.head())
print(df["label"].value_counts().head())

Processing classes:   0%|          | 0/101 [00:00<?, ?it/s]

Dataframe shape: (101000, 2)
                                            filepath      label
0  /kaggle/input/datasets/dansbecker/food-101/foo...  apple_pie
1  /kaggle/input/datasets/dansbecker/food-101/foo...  apple_pie
2  /kaggle/input/datasets/dansbecker/food-101/foo...  apple_pie
3  /kaggle/input/datasets/dansbecker/food-101/foo...  apple_pie
4  /kaggle/input/datasets/dansbecker/food-101/foo...  apple_pie
label
apple_pie         1000
baby_back_ribs    1000
baklava           1000
beef_carpaccio    1000
beef_tartare      1000
Name: count, dtype: int64


In [47]:
label_to_index = {label: idx for idx, label in enumerate(class_names)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

df["label_idx"] = df["label"].map(label_to_index)

print(df.head())
print("Number of classes:", len(label_to_index))

                                            filepath      label  label_idx
0  /kaggle/input/datasets/dansbecker/food-101/foo...  apple_pie          0
1  /kaggle/input/datasets/dansbecker/food-101/foo...  apple_pie          0
2  /kaggle/input/datasets/dansbecker/food-101/foo...  apple_pie          0
3  /kaggle/input/datasets/dansbecker/food-101/foo...  apple_pie          0
4  /kaggle/input/datasets/dansbecker/food-101/foo...  apple_pie          0
Number of classes: 101


In [48]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label_idx"],
    random_state=42
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

Train shape: (80800, 3)
Validation shape: (20200, 3)


In [49]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["filepath"].values, train_df["label_idx"].values)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["filepath"].values, val_df["label_idx"].values)
)

train_ds = (
    train_ds
    .shuffle(1000)
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(train_ds)
print(val_ds)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


In [50]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

Epoch 1/3


I0000 00:00:1776162834.697101     129 service.cc:152] XLA service 0x78ee88003ff0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776162834.697154     129 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776162834.697161     129 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776162836.951925     129 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-14 10:34:04.096004: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-14 10:34:04.239707: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-14 10:34:04.579480: E external/local_xl

2525/2525 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.0098 - loss: 4.7065

2026-04-14 10:38:29.392717: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-14 10:38:29.529715: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-14 10:38:29.832353: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-14 10:38:29.972120: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-14 10:38:30.719803: E external/local_xla/xla/stream_

2525/2525 ━━━━━━━━━━━━━━━━━━━━ 289s 104ms/step - accuracy: 0.0098 - loss: 4.7065 - val_accuracy: 0.0099 - val_loss: 4.6960
Epoch 2/3
2525/2525 ━━━━━━━━━━━━━━━━━━━━ 123s 49ms/step - accuracy: 0.0102 - loss: 4.7050 - val_accuracy: 0.0099 - val_loss: 4.6746
Epoch 3/3
2275/2525 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - accuracy: 0.0098 - loss: 4.7047

KeyboardInterrupt: 

In [52]:
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 101)            │       129,381 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,178,952 (15.94 MB)

 Trainable params: 129,381 (505.39 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [58]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=2,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1
    )
]